# CreditFlow P4 — Model Benchmark

Benchmark 4 models for credit risk prediction.

**Business Objective**: Optimize Recall/F1 to detect defaults.

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
print('Setup complete')

In [ ]:
# Configuration
RANDOM_STATE = 42
N_SAMPLES = 5000
DEFAULT_RATE = 0.12
TEST_SIZE = 0.15
VAL_SIZE = 0.15
np.random.seed(RANDOM_STATE)
print(f'Config: {N_SAMPLES} samples, {DEFAULT_RATE:.0%} default rate')

In [ ]:
# Generate synthetic credit data
def generate_credit_data(n_samples, default_rate, random_state):
    np.random.seed(random_state)
    n_defaults = int(n_samples * default_rate)
    n_non_defaults = n_samples - n_defaults
    non_defaults = pd.DataFrame({
        'income': np.random.lognormal(8.0, 0.5, n_non_defaults),
        'age': np.random.normal(40, 10, n_non_defaults).clip(18, 80).astype(int),
        'employment_years': np.random.exponential(5, n_non_defaults).clip(0, 40),
        'loan_amount': np.random.lognormal(9.0, 0.6, n_non_defaults),
        'loan_term': np.random.choice([12, 24, 36, 48, 60], n_non_defaults),
        'existing_debt': np.random.exponential(2000, n_non_defaults),
        'credit_history': np.random.exponential(5, n_non_defaults).clip(0, 30),
        'previous_defaults': np.random.poisson(0.2, n_non_defaults),
        'default': 0
    })
    defaults = pd.DataFrame({
        'income': np.random.lognormal(7.5, 0.6, n_defaults),
        'age': np.random.normal(35, 12, n_defaults).clip(18, 80).astype(int),
        'employment_years': np.random.exponential(3, n_defaults).clip(0, 40),
        'loan_amount': np.random.lognormal(9.5, 0.7, n_defaults),
        'loan_term': np.random.choice([12, 24, 36, 48, 60, 72], n_defaults),
        'existing_debt': np.random.exponential(5000, n_defaults),
        'credit_history': np.random.exponential(3, n_defaults).clip(0, 30),
        'previous_defaults': np.random.poisson(1.5, n_defaults),
        'default': 1
    })
    df = pd.concat([non_defaults, defaults], ignore_index=True)
    return df.sample(frac=1, random_state=random_state).reset_index(drop=True)

df = generate_credit_data(N_SAMPLES, DEFAULT_RATE, RANDOM_STATE)
print(f'Shape: {df.shape}, Default rate: {df["default"].mean():.2%}')
df.head()

In [ ]:
# Add derived features
def add_derived_features(df):
    out = df.copy()
    out['debt_to_income'] = out['existing_debt'] / out['income'].replace(0, np.nan)
    out['loan_to_income'] = out['loan_amount'] / out['income'].replace(0, np.nan)
    out['debt_to_loan'] = out['existing_debt'] / out['loan_amount'].replace(0, np.nan)
    out['employment_stability'] = (out['employment_years'] / 10).clip(upper=1.0)
    age_minus_18 = (out['age'] - 18).clip(lower=1)
    out['credit_history_year_ratio'] = out['credit_history'] / age_minus_18
    return out

df = add_derived_features(df)
print(f'Features added. Shape: {df.shape}')

In [ ]:
# Train/Val/Test split
FEATURES = ['income', 'age', 'employment_years', 'loan_amount', 'loan_term',
            'existing_debt', 'credit_history', 'previous_defaults',
            'debt_to_income', 'loan_to_income', 'debt_to_loan',
            'employment_stability', 'credit_history_year_ratio']
TARGET = 'default'
X = df[FEATURES]
y = df[TARGET]
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=(TEST_SIZE + VAL_SIZE), random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, random_state=RANDOM_STATE, stratify=y_tmp
)
print(f'Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}')

In [ ]:
# Define models
models = [
    ('logistic_regression', LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'
    )),
    ('decision_tree', DecisionTreeClassifier(
        max_depth=5, random_state=RANDOM_STATE, class_weight='balanced'
    )),
    ('random_forest', RandomForestClassifier(
        n_estimators=100, max_depth=5, random_state=RANDOM_STATE, class_weight='balanced'
    )),
]
if XGBOOST_AVAILABLE:
    models.append(('xgboost', XGBClassifier(
        n_estimators=100, max_depth=5, random_state=RANDOM_STATE,
        use_label_encoder=False, eval_metric='logloss', scale_pos_weight=5
    )))
print(f'Models: {[n for n, _ in models]}')

In [ ]:
# Train and evaluate
results = []
for name, model in models:
    print(f'Training {name}...')
    pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
    pipe.fit(X_train, y_train)
    y_prob = pipe.predict_proba(X_val)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    metrics = {
        'model': name,
        'accuracy': accuracy_score(y_val, y_pred),
        'precision': precision_score(y_val, y_pred, zero_division=0),
        'recall': recall_score(y_val, y_pred, zero_division=0),
        'f1': f1_score(y_val, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_val, y_prob)
    }
    results.append(metrics)
    print(f'  F1: {metrics["f1"]:.4f}, Recall: {metrics["recall"]:.4f}')
print('Done!')

In [ ]:
# Results table
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('f1', ascending=False).reset_index(drop=True)
print('=' * 60)
print('MODEL BENCHMARK (sorted by F1)')
print('=' * 60)
print(results_df.round(4).to_string(index=False))
print('\nBest by F1:', results_df.iloc[0]['model'])

In [ ]:
# Visualize confusion matrices
fig, axes = plt.subplots(1, len(results), figsize=(5*len(results), 4))
if len(results) == 1:
    axes = [axes]
for ax, (name, model) in zip(axes, models):
    pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_val)
    cm = confusion_matrix(y_val, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Default', 'Default'],
                yticklabels=['No Default', 'Default'])
    ax.set_title(name)
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
plt.suptitle('Confusion Matrices (Validation Set)', fontsize=14)
plt.tight_layout()
plt.show()